# 01 Data Sources, Data Imports, and Assets

Materialize source data, write the source manifest/import spec, optionally run `az ml data import`, and register the raw Azure ML data asset.


## 1) Notebook Dependency Bootstrap

Validate the packages this notebook needs before importing the pipeline dependencies.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "azure-ai-ml": "azure.ai.ml",
    "azure-ai-projects": "azure.ai.projects",
    "azure-identity": "azure.identity",
    "python-dotenv": "dotenv",
    "kaggle": "kaggle",
    "mltable": "mltable",
}

def module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

missing_packages = [
    package_name
    for package_name, module_name in REQUIRED_PACKAGES.items()
    if not module_available(module_name)
]

if missing_packages:
    print("Installing missing packages with the active notebook interpreter:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing_packages,
    ])
    importlib.invalidate_caches()
else:
    print("All required packages are already available in this notebook environment.")

print("Python executable:", sys.executable)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import fetch_openml

candidate_src_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "aml-foundry-integration" / "src",
    Path.cwd().parent.parent / "aml-foundry-integration" / "src",
]
for package_src in candidate_src_paths:
    if (package_src / "aml_bandits" / "settings.py").exists():
        if str(package_src) not in sys.path:
            sys.path.insert(0, str(package_src))
        break

RNG = np.random.default_rng(42)
pd.set_option("display.max_columns", 120)


## 2) Azure ML Configuration

Use environment variables first. The resource group and workspace defaults match this local demo workspace and do not include secrets.

In [ ]:
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import DefaultAzureCredential

from aml_bandits.settings import (
    apply_project_environment_defaults,
    load_project_defaults,
    project_summary,
)

load_dotenv()
PROJECT_DEFAULTS = apply_project_environment_defaults(load_project_defaults())
PROJECT_BINDING = project_summary(PROJECT_DEFAULTS)

DEFAULT_AZUREML_RESOURCE_GROUP = PROJECT_DEFAULTS.azure_ml.resource_group
DEFAULT_AZUREML_WORKSPACE_NAME = PROJECT_DEFAULTS.azure_ml.workspace_name
DEFAULT_AZUREML_LOCATION = PROJECT_DEFAULTS.azure_ml.location

AZUREML_SUBSCRIPTION_ID = (
    os.getenv("AZUREML_SUBSCRIPTION_ID")
    or os.getenv("AZURE_SUBSCRIPTION_ID")
    or os.getenv("SUBSCRIPTION_ID")
    or ""
)
AZUREML_RESOURCE_GROUP = os.getenv("AZUREML_RESOURCE_GROUP", DEFAULT_AZUREML_RESOURCE_GROUP)
AZUREML_WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", DEFAULT_AZUREML_WORKSPACE_NAME)
AZUREML_LOCATION = os.getenv("AZUREML_LOCATION", DEFAULT_AZUREML_LOCATION)
AZUREML_DATA_ASSET_NAME = os.getenv("AZUREML_DATA_ASSET_NAME", "bank-marketing-7mlet")
AZUREML_DATA_ASSET_VERSION = os.getenv("AZUREML_DATA_ASSET_VERSION", "1")

INGESTION_DIR = Path(
    os.getenv(
        "BANK_MARKETING_INGESTION_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "azureml_ingestion"),
    )
).resolve()
INGESTION_DIR.mkdir(parents=True, exist_ok=True)
os.environ["BANK_MARKETING_INGESTION_DIR"] = str(INGESTION_DIR)

AZUREML_SPEC_DIR = Path(
    os.getenv(
        "AZUREML_SPEC_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "azureml_specs"),
    )
).resolve()
AZUREML_SPEC_DIR.mkdir(parents=True, exist_ok=True)

print("Azure ML resource group:", AZUREML_RESOURCE_GROUP)
print("Azure ML workspace:", AZUREML_WORKSPACE_NAME)
print("Azure ML location:", AZUREML_LOCATION)
print("Subscription configured:", bool(AZUREML_SUBSCRIPTION_ID))
print("Foundry project endpoint:", os.getenv("FOUNDRY_PROJECT_ENDPOINT"))
print("Foundry feature agent:", os.getenv("FOUNDRY_FEATURE_AGENT_NAME"))
print("Foundry strategy agent:", os.getenv("FOUNDRY_STRATEGY_AGENT_NAME"))
print("Available Foundry agents:", os.getenv("FOUNDRY_AVAILABLE_AGENTS"))
print("Local ingestion folder:", INGESTION_DIR)
print("Azure ML spec folder:", AZUREML_SPEC_DIR)


## 3) Materialize Source Data for Ingestion

Kaggle, local CSVs, or OpenML are copied or written into one local ingestion folder before the pipeline reads any rows.

In [ ]:
def _try_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists() or path.suffix.lower() != ".csv":
        return None
    for sep in [",", ";", "\t", "|"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 2:
                return df
        except Exception:
            continue
    return None


def _find_target_column(columns: Sequence[str]) -> Optional[str]:
    lowered = {c.lower(): c for c in columns}
    candidates = [
        "y",
        "target",
        "label",
        "subscribed",
        "conversion",
        "converted",
        "response",
        "class",
    ]
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    return None


def _csv_files(root: Path) -> List[Path]:
    if root.is_file() and root.suffix.lower() == ".csv":
        return [root]
    if not root.exists():
        return []
    return sorted(path for path in root.rglob("*.csv") if path.is_file())


def _download_kaggle_csvs(destination: Path) -> List[Path]:
    datasets = [
        "henriqueyamahata/bank-marketing",
        "tunguz/bank-marketing-data-set",
        "dharmik34/bank-term-deposit-subscription",
    ]
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
    except Exception as ex:
        print(f"Kaggle materialization skipped: {ex}")
        return []

    for dataset_name in datasets:
        try:
            api.dataset_download_files(
                dataset_name,
                path=str(destination),
                unzip=True,
                quiet=True,
            )
            csvs = _csv_files(destination)
            if csvs:
                return csvs
        except Exception as ex:
            print(f"Kaggle dataset skipped ({dataset_name}): {ex}")
    return []


def _copy_local_csvs(repo_root: Path, destination: Path) -> List[Path]:
    candidates = [
        repo_root / "tmp" / "7mlet" / "bank-additional-full.csv",
        repo_root / "tmp" / "7mlet" / "bank-full.csv",
        repo_root / "tmp" / "7mlet" / "bank.csv",
        repo_root / "data" / "assets" / "bank-additional-full.csv",
        repo_root / "data" / "assets" / "bank-full.csv",
    ]

    copied: List[Path] = []
    for source in candidates:
        if not source.exists():
            continue
        target = destination / source.name
        if source.resolve() != target.resolve():
            shutil.copy2(source, target)
        copied.append(target)
    return copied


def _materialize_openml_csv(destination: Path) -> List[Path]:
    try:
        bunch = fetch_openml(name="bank-marketing", version=1, as_frame=True, parser="auto")
    except TypeError:
        bunch = fetch_openml(name="bank-marketing", version=1, as_frame=True)

    output_path = destination / "bank_marketing_openml.csv"
    bunch.frame.copy().to_csv(output_path, index=False)
    return [output_path]


def materialize_bank_marketing_csvs() -> Tuple[List[Path], str]:
    repo_root = Path.cwd()
    INGESTION_DIR.mkdir(parents=True, exist_ok=True)

    existing_csvs = _csv_files(INGESTION_DIR)
    if existing_csvs:
        return existing_csvs, "existing local ingestion folder"

    kaggle_csvs = _download_kaggle_csvs(INGESTION_DIR)
    if kaggle_csvs:
        return kaggle_csvs, "Kaggle API materialized to local ingestion folder"

    local_csvs = _copy_local_csvs(repo_root, INGESTION_DIR)
    if local_csvs:
        return local_csvs, "local CSVs copied to ingestion folder"

    openml_csvs = _materialize_openml_csv(INGESTION_DIR)
    return openml_csvs, "OpenML materialized to local ingestion folder"


materialized_csvs, materialization_provenance = materialize_bank_marketing_csvs()
print(f"Materialized source: {materialization_provenance}")
print(f"Ingestion folder: {INGESTION_DIR}")
print("CSV files:", [path.name for path in materialized_csvs])

## 4) Prepare Azure ML Data Sources and Data Imports

Write a source manifest and Azure ML data import specification before registering assets. The data import command is generated every time and only executed when `ENABLE_AZUREML_DATA_IMPORT=true`.


In [ ]:
DATA_SOURCE_SPEC_DIR = AZUREML_SPEC_DIR / "data_sources"
DATA_SOURCE_SPEC_DIR.mkdir(parents=True, exist_ok=True)

ENABLE_AZUREML_DATA_IMPORT = os.getenv("ENABLE_AZUREML_DATA_IMPORT", "false").strip().lower() in {"1", "true", "yes", "y"}
RAW_DATA_SOURCE_NAME = os.getenv("AZUREML_RAW_DATA_SOURCE_NAME", f"{AZUREML_DATA_ASSET_NAME}-source")

source_manifest = {
    "name": RAW_DATA_SOURCE_NAME,
    "source_type": materialization_provenance,
    "local_landing_folder": str(INGESTION_DIR),
    "csv_files": [path.name for path in materialized_csvs],
    "azure_ml": {
        "resource_group": AZUREML_RESOURCE_GROUP,
        "workspace_name": AZUREML_WORKSPACE_NAME,
        "data_asset_name": AZUREML_DATA_ASSET_NAME,
        "data_asset_version": AZUREML_DATA_ASSET_VERSION,
    },
    "foundry_project": PROJECT_BINDING["foundry"],
    "privacy": "raw_rows_stay_in_azure_ml_data_layer",
}

source_manifest_path = DATA_SOURCE_SPEC_DIR / "raw_data_source_manifest.json"
source_manifest_path.write_text(
    json.dumps(source_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

raw_data_import_spec_path = DATA_SOURCE_SPEC_DIR / "raw_data_import.yml"
raw_data_import_spec_path.write_text(
    "\n".join(
        [
            "$schema: https://azuremlschemas.azureedge.net/latest/data.schema.json",
            f"name: {AZUREML_DATA_ASSET_NAME}",
            f"version: {AZUREML_DATA_ASSET_VERSION}",
            "type: uri_folder",
            "description: 7-MLET raw bank marketing source imported into Azure ML from the notebook landing folder.",
            "tags:",
            "  stage: raw_ingestion",
            "  source_manifest: raw_data_source_manifest.json",
            "  foundry_project: miq-project-miqsec26",
            "  default_agent: finance-orchestrator",
            f"path: {INGESTION_DIR.as_posix()}",
            "",
        ]
    ),
    encoding="utf-8",
)

data_import_command = [
    "az",
    "ml",
    "data",
    "import",
    "--file",
    str(raw_data_import_spec_path),
    "--resource-group",
    AZUREML_RESOURCE_GROUP,
    "--workspace-name",
    AZUREML_WORKSPACE_NAME,
    "--skip-validation",
]

print("Data source manifest:", source_manifest_path)
print("Data import spec:", raw_data_import_spec_path)
print("Data import command:", " ".join(data_import_command))

if ENABLE_AZUREML_DATA_IMPORT:
    completed = subprocess.run(data_import_command, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        print("Data import did not complete; SDK data asset registration remains the fallback path.")
else:
    print("Data import execution is gated. Set ENABLE_AZUREML_DATA_IMPORT=true to materialize through az ml data import.")


## 5) Register Azure ML Data Asset

Create or update a URI_FOLDER data asset from the local ingestion folder. This is separate from later Foundry agent validation, and raw customer rows are not sent to agents.

In [ ]:
from datetime import datetime, timezone

def build_ml_client() -> Optional[MLClient]:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    if AZUREML_SUBSCRIPTION_ID:
        return MLClient(
            credential=credential,
            subscription_id=AZUREML_SUBSCRIPTION_ID,
            resource_group_name=AZUREML_RESOURCE_GROUP,
            workspace_name=AZUREML_WORKSPACE_NAME,
        )

    try:
        return MLClient.from_config(credential=credential)
    except Exception as ex:
        print("Azure ML client not created. Set AZUREML_SUBSCRIPTION_ID or provide an Azure ML config file.")
        print(f"Details: {ex}")
        return None


def create_or_update_data_asset(ml_client: MLClient, data_folder: Path) -> Data:
    def data_asset(version: str) -> Data:
        return Data(
            name=AZUREML_DATA_ASSET_NAME,
            version=version,
            type=AssetTypes.URI_FOLDER,
            path=str(data_folder),
            description="7-MLET bank marketing ingestion folder for contextual bandit demo.",
        )

    try:
        return ml_client.data.create_or_update(data_asset(AZUREML_DATA_ASSET_VERSION))
    except Exception as ex:
        fallback_version = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
        print(f"Data asset version {AZUREML_DATA_ASSET_VERSION!r} was not updated: {ex}")
        print(f"Retrying with generated version {fallback_version}.")
        return ml_client.data.create_or_update(data_asset(fallback_version))


ml_client = build_ml_client()
registered_data_asset = None

if ml_client is not None:
    registered_data_asset = create_or_update_data_asset(ml_client, INGESTION_DIR)
    os.environ["AZUREML_REGISTERED_DATA_ASSET_ID"] = registered_data_asset.id
    print("Registered Azure ML data asset:", registered_data_asset.name)
    print("Version:", registered_data_asset.version)
    print("Asset id:", registered_data_asset.id)
else:
    print("Continuing with the local ingestion folder:", INGESTION_DIR)

print("For Azure ML jobs, mount or download this asset and set AZUREML_DATA_ASSET_PATH to the local mount path.")